# *Om Sai Ram*

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [23]:
df=pd.read_csv("train_dataset.csv")

In [24]:
df.shape

(150, 6)

In [25]:
df.sample(5)

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield
113,65.61,2.31,436.02,2.33,432.37,46.809
76,62.85,3.91,476.73,11.54,372.41,7.473
67,65.16,3.11,485.80,20.64,382.14,0.290
51,63.13,0.79,478.47,14.33,457.42,0.000
147,42.70,1.26,463.01,5.96,458.71,0.645


In [27]:
df["residence_index"]=df["length_m"]/df["flow_rate_L_min"]

In [28]:
# Try to find the approximate cutoff visually — looks like ~450K for inlet, ~460K for jacket
df['above_temp_threshold'] = ((df['inlet_temperature_K'] > 450) | (df['jacket_temperature_K'] > 460)).astype(int)

In [29]:
df.sample(5)

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield,residence_index,above_temp_threshold
136,43.91,2.13,360.91,3.74,428.49,14.583,0.085174,0
109,10.77,0.84,363.69,16.27,486.30,0.000,1.510678,1
73,66.16,1.79,452.45,21.37,454.90,0.000,0.323005,1
149,25.90,0.80,362.53,2.94,474.85,28.516,0.113514,1
69,79.02,3.47,360.40,5.46,407.02,10.824,0.069096,0


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   flow_rate_L_min       150 non-null    float64
 1   concentration_mol_L   150 non-null    float64
 2   inlet_temperature_K   150 non-null    float64
 3   length_m              150 non-null    float64
 4   jacket_temperature_K  150 non-null    float64
 5   overall_yield         150 non-null    float64
 6   residence_index       150 non-null    float64
 7   above_temp_threshold  150 non-null    int64  
dtypes: float64(7), int64(1)
memory usage: 9.5 KB


In [31]:
df.describe()

,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield,residence_index,above_temp_threshold
count,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000,150.000000
mean,40.466800,2.311333,424.423400,14.086533,442.240800,36.222173,0.567448,0.580000
std,22.239772,1.019844,45.200641,6.995343,55.047768,38.427427,0.620900,0.495212
min,5.410000,0.520000,351.630000,2.260000,354.010000,0.000000,0.035513,0.000000
25%,21.112500,1.362500,387.255000,8.027500,393.327500,0.013500,0.191077,0.000000
50%,38.610000,2.445000,425.630000,14.235000,437.080000,15.314000,0.347751,1.000000
75%,61.227500,3.155000,462.972500,20.685000,486.180000,74.876500,0.694635,1.000000
max,79.020000,3.970000,498.580000,24.990000,547.990000,99.971000,4.493530,1.000000


In [32]:
df.duplicated().sum()

np.int64(0)

In [34]:
#Higher MI = more useful feature; MI ≈ 0 = little predictive value.

from sklearn.feature_selection import mutual_info_regression

x_try = df.drop(columns=['overall_yield'])
y = df['overall_yield']

mi_scores = mutual_info_regression(x_try, y, random_state=42)
mi_series = pd.Series(mi_scores, index=x_try.columns).sort_values(ascending=False)
print(mi_series)

residence_index         0.339039
above_temp_threshold    0.315311
inlet_temperature_K     0.196524
jacket_temperature_K    0.189365
flow_rate_L_min         0.099079
length_m                0.093101
concentration_mol_L     0.000000
dtype: float64


In [35]:
x= df[['residence_index', 'above_temp_threshold','inlet_temperature_K', 'jacket_temperature_K']]

In [43]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [44]:
from sklearn.ensemble import GradientBoostingRegressor
gbr=GradientBoostingRegressor(n_estimators=300, random_state=42, max_depth=3)

In [45]:
gbr.fit(x_train,y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",300
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft 

In [46]:
results=gbr.predict(x_test)

In [48]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)
print("Gradient Boosting Regressor")
print("R² Score :", r2_score(y_test, results))
print("MAE      :", mean_absolute_error(y_test, results))
print("MSE      :", mean_squared_error(y_test, results))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, results)))
print("MAPE     :", mean_absolute_percentage_error(y_test, results))

Gradient Boosting Regressor
R² Score : 0.8522141559912474
MAE      : 10.61054364376401
MSE      : 259.8670752451834
RMSE     : 16.120393147972024
MAPE     : 1.0758442611680742e+16


In [50]:
#import joblib

# Save the trained Gradient Boosting model
#joblib.dump(gbr, "gradient_boosting_model.pkl")

#print("Model saved successfully!")

Model saved successfully!


## Model Summary & Explanation

**Data:** 150 clean rows (no nulls/duplicates), weak linear correlations with target → hinted the real relationship isn't linear.

**Insight:** Scatter plots revealed a hard temperature threshold — yield is noisy below ~450K, collapses near 0 above it. Classic reaction-kinetics cutoff (catalyst deactivation / side reaction dominance).

**Features kept:**
- `residence_index` (engineered = `length_m / flow_rate_L_min`) — strongest mutual information signal; reaction residence time matters more than flow or length alone
- `above_temp_threshold` (engineered binary flag) — directly encodes the observed cutoff
- `inlet_temperature_K`, `jacket_temperature_K` (raw) — still useful within each regime

**Features dropped:**
- `concentration_mol_L` — zero mutual information, irrelevant
- Raw `flow_rate_L_min`, `length_m` — redundant with `residence_index`; re-adding them measurably hurt performance
- `temp_difference`, `heat_exposure`, `thermal_reactivity`, `flow_concentration` — collinear with parent features (VIF = inf / near-perfect correlation)

**Model:** Gradient Boosting Regressor — tree-based models split on thresholds naturally, matching the data's step-function behavior. Outperformed Random Forest and XGBoost (both under- or over-fit on this small dataset).

**Parameters:**
- `max_depth=3` — kept shallow to avoid memorizing noise with only ~120 training rows
- `n_estimators=300` — enough trees to average out noise without overfitting